# Credit card fraud detection: a machine learning approach to imbalanced binary classification

---

Dataset: [Kaggle — Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
Author: Tersi Yousra

Programme: Exploratory data analysis

---

### Abstract

Credit card fraud represents a significant and growing problem in the financial sector. According to the Nilson Report, global card fraud losses exceeded \$33 billion in 2022, underscoring the critical need for robust automated detection systems. This notebook presents a rigorous end-to-end machine learning pipeline applied to a real-world transaction dataset, addressing the fundamental challenge of extreme class imbalance? a defining characteristic of fraud detection that invalidates naive modelling approaches. The analysis proceeds through exploratory data analysis to characterise the dataset and surface latent patterns, followed by pre-processing and feature engineering, class imbalance handling via SMOTE and class-weight calibration, model training and hyperparameter tuning, and finally rigorous evaluation using business-appropriate metrics.

### Why accuracy is the wrong metric

With only ~0.17% of transactions being fraudulent, a trivial classifier that labels *every* transaction as legitimate achieves 99.83% accuracy whilst providing zero utility. This is the accuracy paradox: in the presence of severe class imbalance, accuracy conflates the majority class's overwhelming representation with genuine predictive power. The appropriate metrics for this domain are recall (sensitivity), which measures what fraction of actual frauds were caught and where misses carry direct financial and reputational costs; precision, which measures how many of the flagged transactions were genuinely fraudulent and where false positives erode customer trust and incur manual review costs; the F1-score, which is the harmonic mean of precision and recall providing a single balanced summary; and the area under the precision-recall curve (AUPRC), which is preferred over ROC-AUC in highly imbalanced settings because it is not inflated by the large number of true negatives.

---
## Section 0:

All required libraries are imported here and the random seed is configured for full reproducibility. Setting `RANDOM_STATE` globally ensures that stochastic components (train/test splitting, SMOTE oversampling, and tree-based model initialisation) all produce identical results across runs.

In [ ]:
# ── Standard library 
import warnings
warnings.filterwarnings('ignore')   
# ── Core data manipulation 
import numpy as np
import pandas as pd

# ── Visualisation 
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── pre-processing & model selection 
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import RobustScaler

# ── SMOTE 
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# ── Models 
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression  

# ── Evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    f1_score,
)

# ── Global configuration
RANDOM_STATE = 42  
np.random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Environment configured. All libraries loaded successfully.')

---
## Section 1:

The dataset used in this analysis is the ULB Credit Card Fraud Detection dataset, published by the Machine Learning Group at Université Libre de Bruxelles. It contains 284,807 credit card transactions made in September 2013 by European cardholders over two days. Features `V1` through `V28` are the result of a Principal Component Analysis (PCA) transformation applied by the original authors to protect cardholder privacy, meaning the semantic meaning of individual components is not interpretable in the conventional sense. The `Time` column records seconds elapsed since the first transaction in the dataset, `Amount` gives the transaction value in Euros, and `Class` is the target variable where 0 denotes a legitimate transaction and 1 denotes fraud.

In [ ]:
df = pd.read_csv('creditcard.csv')

print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df.head()

In [ ]:
df.info()

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'None — dataset is complete.')

In [ ]:
df[['Time', 'Amount', 'Class']].describe().round(4)

---
## Section 2:

EDA serves a dual purpose. It builds domain intuition about the data distribution, and it surfaces potential issues (outliers, skew, multicollinearity) that must be addressed before modelling. Given that the PCA features are anonymised, the analysis focuses on quantifying and visualising the class imbalance, characterising the `Amount` and `Time` distributions across classes, and examining the correlation structure of the PCA components.

### 2.1 Class distribution

In [ ]:
class_counts = df['Class'].value_counts()
fraud_pct = class_counts[1] / len(df) * 100

print(f'Legitimate transactions : {class_counts[0]:>7,}  ({100 - fraud_pct:.4f}%)')
print(f'Fraudulent transactions : {class_counts[1]:>7,}  ({fraud_pct:.4f}%)')
print(f'Imbalance ratio         : {class_counts[0] / class_counts[1]:.0f}:1')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bars = axes[0].bar(['Legitimate (0)', 'Fraud (1)'],
                   class_counts.values,
                   color=['#4C72B0', '#DD8452'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Transaction Class Counts (Absolute)', fontweight='bold')
axes[0].set_ylabel('Number of Transactions')
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{count:,}', ha='center', va='bottom', fontsize=10)

axes[1].bar(['Legitimate (0)', 'Fraud (1)'],
            class_counts.values,
            color=['#4C72B0', '#DD8452'], edgecolor='white', linewidth=1.5)
axes[1].set_yscale('log')
axes[1].set_title('Transaction Class Counts (Log Scale)', fontweight='bold')
axes[1].set_ylabel('Number of Transactions (log₁₀)')

plt.suptitle('Extreme Class Imbalance: ~575 Legitimate per 1 Fraudulent Transaction',
             y=1.02, fontsize=12, color='#555')
plt.tight_layout()
plt.show()

### 2.2 Transaction amount distribution by class

A meaningful question is whether fraudulent transactions differ systematically in their monetary value from legitimate ones. If fraudsters tend to transact at atypical amounts, the `Amount` feature may carry discriminative signal beyond the PCA components.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

legit = df[df['Class'] == 0]['Amount']
fraud = df[df['Class'] == 1]['Amount']

axes[0].hist(legit, bins=100, alpha=0.6, color='#4C72B0',
             label=f'Legitimate (n={len(legit):,})', density=True, log=True)
axes[0].hist(fraud, bins=50, alpha=0.7, color='#DD8452',
             label=f'Fraud (n={len(fraud):,})', density=True, log=True)
axes[0].set_xlabel('Transaction Amount (€)')
axes[0].set_ylabel('Density (log scale)')
axes[0].set_title('Amount Distribution by Class', fontweight='bold')
axes[0].legend()

amount_df = df[['Amount', 'Class']].copy()
amount_df['Class_label'] = amount_df['Class'].map({0: 'Legitimate', 1: 'Fraud'})
sns.boxplot(data=amount_df, x='Class_label', y='Amount',
            palette={'Legitimate': '#4C72B0', 'Fraud': '#DD8452'},
            showfliers=False, ax=axes[1])  # hide outliers for readability
axes[1].set_title('Amount Distribution — Box Plot (outliers hidden)', fontweight='bold')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Transaction Amount (€)')

plt.tight_layout()
plt.show()

print(f"Legitimate — Median: €{legit.median():.2f}  |  Mean: €{legit.mean():.2f}  |  Max: €{legit.max():.2f}")
print(f"Fraud      — Median: €{fraud.median():.2f}  |  Mean: €{fraud.mean():.2f}  |  Max: €{fraud.max():.2f}")

### 2.3 Transaction time distribution

The `Time` feature records elapsed seconds since the first transaction. A temporal analysis can reveal whether fraud attempts cluster at specific periods, for instance, late-night hours when cardholder vigilance or bank monitoring may be reduced.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

df['Hour'] = (df['Time'] / 3600).astype(int)

legit_by_hour = df[df['Class'] == 0].groupby('Hour').size()
fraud_by_hour = df[df['Class'] == 1].groupby('Hour').size()

ax.fill_between(legit_by_hour.index, legit_by_hour.values,
                alpha=0.4, color='#4C72B0', label='Legitimate')
ax.plot(legit_by_hour.index, legit_by_hour.values, color='#4C72B0', linewidth=1.5)

ax2 = ax.twinx()
ax2.bar(fraud_by_hour.index, fraud_by_hour.values,
        alpha=0.6, color='#DD8452', label='Fraud (right axis)')
ax2.set_ylabel('Fraudulent Transactions per Hour', color='#DD8452')
ax2.tick_params(axis='y', labelcolor='#DD8452')

ax.set_xlabel('Hour of Observation Period')
ax.set_ylabel('Legitimate Transactions per Hour', color='#4C72B0')
ax.tick_params(axis='y', labelcolor='#4C72B0')
ax.set_title('Transaction Volume Over Time (Legitimate vs Fraud)', fontweight='bold')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

### 2.4 PCA feature distributions

Although the PCA features are anonymous, examining their distributions by class can identify which components carry the greatest discriminative power. Features with markedly different distributions between classes are likely to be the most informative predictors.

In [ ]:
pca_features = [f'V{i}' for i in range(1, 13)]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, feature in enumerate(pca_features):
    for cls, colour, label in [(0, '#4C72B0', 'Legitimate'), (1, '#DD8452', 'Fraud')]:
        subset = df[df['Class'] == cls][feature]
        axes[i].hist(subset, bins=60, alpha=0.55, color=colour,
                     density=True, label=label)
    axes[i].set_title(feature, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Density')
    if i == 0:
        axes[i].legend(fontsize=8)

plt.suptitle('PCA Component Distributions by Class (V1–V12)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 2.5 Correlation heatmap

By definition of PCA, the `V1`–`V28` components are orthogonal and therefore uncorrelated. Any residual correlations visible in the heatmap arise from the non-PCA features (`Time`, `Amount`) or from numerical imprecision. High inter-feature correlation is not expected here, but the check is good practice.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 14))

corr_matrix = df.drop(columns=['Hour']).corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool)) 
sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap='coolwarm',
    vmin=-1, vmax=1,
    center=0,
    annot=False,
    linewidths=0.3,
    ax=ax,
    cbar_kws={'shrink': 0.8, 'label': 'Pearson r'},
)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 3:
### 3.1 Feature scaling

The PCA components `V1`–`V28` are already zero-centred and on comparable scales. However, `Amount` and `Time` have very different magnitudes and distributions. `RobustScaler` is applied to these two features according to the transformation $\tilde{x} = \frac{x - Q_2}{Q_3 - Q_1}$, where $Q_2$ is the median and $Q_3 - Q_1$ is the inter-quartile range. This formulation is more appropriate than `StandardScaler`, which uses mean and standard deviation, because transaction amounts follow a heavy-tailed distribution and standard deviation is highly sensitive to outliers in such settings.

### 3.2 Train / test split

A stratified 80/20 split is used, which preserves the original class ratio in both the training and test sets. This is essential because a non-stratified split risks placing most or all fraud instances into one partition by chance.

In [ ]:
df.drop(columns=['Hour'], inplace=True)

scaler = RobustScaler()
df[['Amount', 'Time']] = scaler.fit_transform(df[['Amount', 'Time']])

X = df.drop(columns=['Class'])
y = df['Class']

print(f'Feature matrix shape : {X.shape}')
print(f'Target vector shape  : {y.shape}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y   
)

print(f'\nTraining set  : {X_train.shape[0]:,} samples '
      f'({y_train.sum():,} fraud, {(y_train == 0).sum():,} legitimate)')
print(f'Test set      : {X_test.shape[0]:,} samples '
      f'({y_test.sum():,} fraud, {(y_test == 0).sum():,} legitimate)')

---
## Section 4:
Class imbalance is not merely a statistical inconvenience; it is a fundamental challenge that systematically biases gradient-based and probability-based learners towards the majority class. Three principal strategies exist. Undersampling removes majority class instances, which is fast and reduces training time but discards potentially useful data. Oversampling via SMOTE synthesises minority class instances, retaining all original data and producing more generalisable models at the cost of potential noise and longer training. Class weighting penalises misclassification of the minority class more heavily without modifying the data, though this may be insufficient under extreme imbalance ratios.

### Synthetic minority over-sampling technique (SMOTE)

SMOTE (Chawla et al., 2002) generates synthetic minority samples by interpolating between existing minority instances in feature space. For each minority instance $x_i$, the algorithm finds its $k$ nearest minority neighbours ($k=5$ by default), selects one neighbour $\hat{x}$ at random, and creates a synthetic sample as $x_{\text{new}} = x_i + \lambda \cdot (\hat{x} - x_i)$ where $\lambda \sim \text{Uniform}(0, 1)$. This is superior to naive random oversampling, which merely duplicates existing instances, because it expands the minority class decision region rather than reinforcing individual points.

SMOTE must be applied only to the training set. Applying it before splitting would cause data leakage , synthetic samples derived from test instances would contaminate the training set, producing optimistically biased evaluation metrics. This constraint is enforced here by applying SMOTE after splitting.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print('Class distribution BEFORE SMOTE:')
counts_before = y_train.value_counts()
print(f'  Legitimate: {counts_before[0]:,}  |  Fraud: {counts_before[1]:,}')
print(f'  Ratio: {counts_before[0]/counts_before[1]:.0f}:1')

print('\nClass distribution AFTER SMOTE:')
counts_after = pd.Series(y_train_resampled).value_counts()
print(f'  Legitimate: {counts_after[0]:,}  |  Fraud: {counts_after[1]:,}')
print(f'  Ratio: {counts_after[0]/counts_after[1]:.2f}:1')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, counts, title in [
    (axes[0], counts_before, 'Before SMOTE (Training Set)'),
    (axes[1], counts_after,  'After SMOTE (Training Set)'),
]:
    ax.bar(['Legitimate', 'Fraud'], counts.values,
           color=['#4C72B0', '#DD8452'], edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 500, f'{v:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

---
## Section 5:

Three classifiers are trained and compared, ranging from a simple linear baseline to a powerful ensemble method. Logistic regression serves as a strong, interpretable baseline whose linear decision boundary is well-understood and provides a lower bound for model performance. Random Forest is a bagged ensemble of decision trees that is robust to overfitting via bootstrap aggregation and naturally handles non-linear decision boundaries. XGBoost is a gradient-boosted ensemble that is generally the strongest performer on tabular data, and key hyperparameters are tuned here to account for class imbalance via `scale_pos_weight`. All models are fitted on the SMOTE-resampled training data and evaluated on the original, unmodified test set.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Model 1: Logistic Regression (Baseline)
# ─────────────────────────────────────────────────────────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                        class_weight='balanced', C=1.0)
lr.fit(X_train_resampled, y_train_resampled)
print('Logistic Regression trained.')

# ─────────────────────────────────────────────────────────────────────────────
# Model 2: Random Forest
# ─────────────────────────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    class_weight='balanced_subsample',  # recompute weights per bootstrap sample
    random_state=RANDOM_STATE,
    n_jobs=-1,   # use all available CPU cores
)
rf.fit(X_train_resampled, y_train_resampled)
print('Random Forest trained.')

# ─────────────────────────────────────────────────────────────────────────────
# Model 3: XGBoost
# ─────────────────────────────────────────────────────────────────────────────
neg_pos_ratio = (y_train_resampled == 0).sum() / (y_train_resampled == 1).sum()

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=neg_pos_ratio,
    eval_metric='aucpr',
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb.fit(X_train_resampled, y_train_resampled,
        eval_set=[(X_test, y_test)],
        verbose=False)
print('XGBoost trained.')

---
## Section 6:

### 6.1 Classification reports

Each model is evaluated on the held-out test set using precision, recall, and F1-score, whose mathematical definitions are given below for completeness.

$$\text{Precision} = \frac{TP}{TP + FP} \qquad \text{Recall} = \frac{TP}{TP + FN} \qquad F_1 = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

where $TP$ denotes true positives, $FP$ false positives, and $FN$ false negatives. In a fraud detection context a false negative (missed fraud) carries a high direct cost because the fraudulent transaction is processed, whereas a false positive (legitimate transaction wrongly flagged) carries an indirect cost through customer friction, manual review overhead, and potential churn. The appropriate weighting between precision and recall is ultimately a business decision that should incorporate the financial magnitude of each error type.

In [ ]:
models = {
    'Logistic Regression': lr,
    'Random Forest':       rf,
    'XGBoost':             xgb,
}

results = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'y_pred': y_pred,
        'y_prob': y_prob,
        'f1':     f1_score(y_test, y_pred),
        'auprc':  average_precision_score(y_test, y_prob),
        'auroc':  roc_auc_score(y_test, y_prob),
    }
    
    print(f'{'='*60}')
    print(f'  {name}')
    print(f'{'='*60}')
    print(classification_report(y_test, y_pred,
                                target_names=['Legitimate', 'Fraud'],
                                digits=4))
    print(f'  AUPRC : {results[name]["auprc"]:.4f}')
    print(f'  AUROC : {results[name]["auroc"]:.4f}\n')

### 6.2 Confusion matrices

The confusion matrix provides the most granular view of classifier performance, decomposing predictions into true negatives, false positives, false negatives, and true positives. In this context, minimising false negatives, the bottom-left cell, is the primary business objective, as these represent fraudulent transactions that pass through undetected.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Legitimate', 'Fraud'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\n(F1={res["f1"]:.4f})', fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.suptitle('Confusion Matrices — All Models on Held-Out Test Set',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 6.3 Precision-recall curves

The precision-recall curve plots precision against recall at every possible classification threshold, and the area under it (AUPRC) summarises this trade-off in a single scalar. AUPRC is preferred over ROC-AUC for imbalanced data because the ROC curve plots the true positive rate against the false positive rate, whose denominator includes the very large number of true negatives. With 184,000+ legitimate transactions in the test set, even a poor classifier can achieve a favourable false positive rate, inflating the ROC-AUC score. The precision-recall curve excludes true negatives from both axes entirely, making it a more honest metric when the minority class is the class of interest.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

colors = ['#4C72B0', '#55A868', '#DD8452']
for (name, res), color in zip(results.items(), colors):
    precision, recall, _ = precision_recall_curve(y_test, res['y_prob'])
    ax.plot(recall, precision, lw=2, color=color,
            label=f"{name} (AUPRC = {res['auprc']:.4f})")

prevalence = y_test.mean()
ax.axhline(y=prevalence, color='grey', linestyle='--', lw=1.5,
           label=f'Random baseline (AUPRC ≈ {prevalence:.4f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — Model Comparison', fontweight='bold', fontsize=13)
ax.legend(fontsize=10, loc='upper right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

### 6.4 Feature importance (Random Forest and XGBoost)

Feature importance scores derived from tree-based models indicate the aggregate contribution of each feature to reducing impurity across all splits. This analysis, while limited by the anonymised nature of the PCA features, is methodologically informative and demonstrates which latent components most strongly discriminate between legitimate and fraudulent transactions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (model, title) in zip(axes, [(rf, 'Random Forest'), (xgb, 'XGBoost')]):
    importances = pd.Series(model.feature_importances_, index=X.columns)
    top_features = importances.nlargest(15).sort_values()
    
    colours = ['#DD8452' if f in ['Amount', 'Time'] else '#4C72B0'
               for f in top_features.index]
    
    top_features.plot(kind='barh', ax=ax, color=colours, edgecolor='white')
    ax.set_title(f'{title} — Top 15 Features by Importance', fontweight='bold')
    ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)')
    ax.set_ylabel('Feature')

plt.tight_layout()
plt.show()

### 6.5 Model comparison summary

In [ ]:
from sklearn.metrics import precision_score, recall_score

summary_rows = []
for name, res in results.items():
    summary_rows.append({
        'Model':     name,
        'Precision': precision_score(y_test, res['y_pred']),
        'Recall':    recall_score(y_test,    res['y_pred']),
        'F1-Score':  res['f1'],
        'AUPRC':     res['auprc'],
        'AUROC':     res['auroc'],
    })

summary_df = pd.DataFrame(summary_rows).set_index('Model')
summary_df = summary_df.round(4)

# Highlight best value per column
summary_df.style.highlight_max(axis=0, color='#c8f0c8').format('{:.4f}')

---
## Section 7:

### Summary of findings

This analysis demonstrated a complete machine learning pipeline for credit card fraud detection, with particular emphasis on the challenges posed by extreme class imbalance. The class imbalance at a ~575:1 ratio invalidates accuracy as a performance metric entirely, since a trivial classifier labelling all transactions as legitimate would achieve over 99.8% accuracy whilst catching zero fraud. SMOTE effectively rebalanced the training set, enabling models to learn a meaningful decision boundary for the minority class, and its application exclusively to training data prevented leakage. Ensemble methods substantially outperformed logistic regression, with XGBoost and Random Forest both achieving strong AUPRC scores, and the gradient-boosted approach of XGBoost was particularly effective at the precision-recall trade-off. PCA components V4, V10, V11, V12, and V14 emerged as the most discriminative features across both tree-based models, suggesting these latent factors capture the most informative transaction characteristics. AUPRC stands as the most honest evaluation metric in this setting because it is not inflated by the correct rejection of the majority class.

### Limitations

The PCA transformation precludes direct feature interpretability, limiting actionable business insights. The dataset is static, whereas real-world fraud patterns exhibit concept drift, fraudsters adapt their behaviour over time, requiring regular model retraining. SMOTE's assumption of linear interpolation between minority instances may also not hold in the true feature space.